# Datos MERRA-2 mediante OPeNDAP de NASA en la nube.

**Requisitos para ejecutar este notebook**

1. Tener una cuenta Earth Data Login
2. Método preferido de autenticación.

**Objetivos**

Usar buenas prácticas de OPeNDAP, [pydap](https://pydap.github.io/pydap/) y xarray para:

- Descubrir todas las URLs OPeNDAP asociadas con una colección MERRA-2.
- Autenticarse mediante EDL (basado en token)
- Explorar la colección MERRA-2 y filtrar variables
- Consolidar metadatos a nivel de colección
- Descargar/transmitir un subconjunto de interés.



`Autor`: Miguel Jimenez-Urias, '25


In [ ]:
from pydap.net import create_session
from pydap.client import get_cmr_urls, consolidate_metadata, open_url
import xarray as xr
import datetime as dt
import earthaccess
import matplotlib.pyplot as plt
import numpy as np
import pydap

In [ ]:
print("xarray version: ", xr.__version__)
print("pydap version: ", pydap.__version__)

### Explorar la colección MERRA-2


In [ ]:
merra2_doi = "10.5067/VJAFPLI1CSIV" # available e.g. GES DISC MERRA-2 documentation 
                                    # https://disc.gsfc.nasa.gov/datasets/M2T1NXSLV_5.12.4/summary?keywords=MERRA-2
# One month of data
time_range=[dt.datetime(2023, 1, 1), dt.datetime(2023, 2, 28)]

In [ ]:
urls = get_cmr_urls(doi=merra2_doi,time_range=time_range, limit=100) # you can increase the limit of results
len(urls)

### Autenticar

Para ocultar la abstracción, usaremos earthaccess para autenticarnos y crear una sesión con caché para consolidar todos los metadatos.


In [ ]:
auth = earthaccess.login(strategy="netrc", persist=True) # you will be promted to add your EDL credentials

# pass Token Authorization to a new Session.
my_session = create_session(session=auth.get_session())


### Explorar variables en la colección y filtrar para conservar solo las deseadas

Hacemos esto especificando que el servidor OPeNDAP de NASA procese solicitudes mediante el protocolo DAP4.

Hay dos formas de hacerlo:

- Usar `pydap` para inspeccionar los metadatos (todas las variables dentro de los archivos y su descripción). Puedes ejecutar el siguiente código para listar todos los nombres de variables.

```python
from pydap.client import open_url
ds = open_url(url, protocol='dap4', session=my_session)
ds.tree() # this will display the entire tree directory

ds[Varname].attributes # will display all the information about Varname in the remote file.
```

- Usar el Data Request Form de OPeNDAP visible desde el navegador. Lo logras tomando una URL OPeNDAP, agregando la extensión `.dmr` y pegando la URL resultante en un navegador.

De cualquiera de las dos formas, puedes inspeccionar nombres de variables y sus descripciones sin descargar arreglos grandes.

Abajo asumimos que sabes qué variables necesitas.


In [ ]:
variables = ['lon', 'lat', 'time', 'T2M', "U2M", "V2M"] # variables of interest
CE = "?dap4.ce="+ "/"+";/".join(variables) # Need to add this string as a query expression to the OPeNDAP URL
new_urls = [url.replace("https", "dap4") + CE for url in urls] # 

### Explorar un archivo remoto individual

Creamos un dataset de Xarray con fines de visualización. Hagamos un gráfico de temperatura cerca de la superficie, en una sola unidad de tiempo.

```{note}
Cada gránulo tiene 24 unidades de tiempo. Fragmentar en tiempo como se hace abajo al generar el dataset asegurará que el rebanado se pase al servidor. Así, el subconjunto ocurre cerca de los datos.
```


In [ ]:
%%time
ds = xr.open_dataset(
    new_urls[0], 
    engine='pydap', 
    session=my_session, 
)
ds

### Visualizar

En `Xarray`, al visualizar datos, los datos se descargan. Al fragmentar en `time` al crear el dataset, aseguramos que, en el momento de visualizar una sola unidad de tiempo en xarray, el rebanado temporal se pase al servidor. Esto reduce la cantidad de datos descargados y acelera la transferencia.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ds['T2M'].isel(time=0).plot();

### Subdividir el dataset agregado mientras se remuestrea en el tiempo

- Seleccionamos datos relevantes para el hemisferio sur, en el área cercana a Sudamérica.
- Almacenamos datos en promedios diarios (a diferencia de datos horarios).

```{note}
Para lograrlo, necesitamos identificar el rebanado en espacio de índices y aplicar `chunk the dataset` antes de transmitir. Hacerlo asegurará que el subconjunto espacial se realice del lado del servidor (OPeNDAP), cerca de los datos, mientras `Xarray` calcula el promedio diario.
```


### Crear agregación de Dataset, fragmentando en el espacio


In [ ]:
%%time

# spatial subset around South America
lat, lon = ds['lat'].data, ds['lon'].data

minLon, maxLon = -100, 50 
iLon = np.where((lon>minLon)&(lon < maxLon))[0]
iLat= np.where(lat < 0)[0]

# Make sure subset is done by server
expected_download = {'lon':len(iLon), 'lat': len(iLat)}

ds = xr.open_mfdataset(
    new_urls,
    engine='pydap', 
    session=my_session,
    concat_dim='time',
    combine='nested',
    parallel=True,
    chunks=expected_download, 
)
ds

### Remuestrear y subdividir
Todas las operaciones en la celda siguiente son perezosas, es decir, se retrasan y no activan cómputo.


In [ ]:
## now subset the Xarray Dataset and rechunk so it is a single chunk
ds = ds.isel(lon=slice(iLon[0], iLon[-1]+1), lat=slice(iLat[0], iLat[-1]+1))

# take daily average and store locally
nds = ds.resample(time="1D").mean()

### Descargar
Almacenar los archivos no solo activará la descarga de los datos, sino también cualquier cómputo diferido anterior, como el remuestreo.


In [ ]:
%%time
nds.to_netcdf("data/Merra2_subset.nc4", mode='w')

### Veamos los datos


In [ ]:
mds = xr.open_dataset("data/Merra2_subset.nc4")
mds

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
mds['T2M'].isel(time=0).plot();